Cell 1 — Cek GPU dan environment

In [ ]:
import torch

print("CUDA Available :", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0))
print("CUDA Torch     :", torch.version.cuda)
print("Torch Version  :", torch.__version__)

Hapus package lama

In [ ]:
!pip uninstall -y mamba-ssm causal-conv1d

Cell 2 — Install Mamba

In [ ]:
!pip install -U pip setuptools wheel ninja packaging
!pip install causal-conv1d>=1.4.0 --no-build-isolation
!pip install mamba-ssm --no-build-isolation

In [ ]:
!pip install mamba-ssm[causal-conv1d] --no-build-isolation

Cell 3 — Import test

# Cell 3 — Import test

In [ ]:
import torch
from mamba_ssm import Mamba

print("Mamba import berhasil")
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

# Cell 4 — Forward test Mamba

In [ ]:
import torch
from mamba_ssm import Mamba

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 8
seq_len = 128
dim = 64

model = Mamba(
    d_model=dim,
    d_state=16,
    d_conv=4,
    expand=2,
).to(device)

x = torch.randn(batch_size, seq_len, dim).to(device)

with torch.no_grad():
    y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)
print("Device      :", device)

# Cell 5 — Mini training test

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from mamba_ssm import Mamba

device = "cuda" if torch.cuda.is_available() else "cpu"

# Dummy data: simulasi sequence network flow
num_samples = 512
seq_len = 64
feature_dim = 32
num_classes = 2

X = torch.randn(num_samples, seq_len, feature_dim)
y = torch.randint(0, num_classes, (num_samples,))

loader = DataLoader(
    TensorDataset(X, y),
    batch_size=32,
    shuffle=True
)

class MambaClassifier(nn.Module):
    def __init__(self, feature_dim, hidden_dim, num_classes):
        super().__init__()
        self.proj = nn.Linear(feature_dim, hidden_dim)
        self.mamba = Mamba(
            d_model=hidden_dim,
            d_state=16,
            d_conv=4,
            expand=2,
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.proj(x)
        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)

model = MambaClassifier(
    feature_dim=feature_dim,
    hidden_dim=64,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(3):
    model.train()
    total_loss = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        output = model(xb)
        loss = criterion(output, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(loader):.4f}")

print("Mini training Mamba selesai.")